<a href="https://colab.research.google.com/github/w4bo/AA2627-unibo-dcai/blob/main/slides/automl-flaml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Learning Outcomes

By the end of this 3-hour lab, you should be able to:

- Explain AutoML as algorithm selection, hyperparameter optimization, and resource allocation.
- Connect FLAML's budget-aware search to the Hutter and Vanschoren AutoML tutorial.
- Run AutoML on real classification and regression datasets.
- Compare AutoML models against manual scikit-learn baselines.
- Study how budget and metric choices change selected models.
- Describe what AutoML automates and what still needs human judgment.

## AutoML in One Picture

![](img/automl/hutter-vanschoren-05.png)

The Hutter and Vanschoren tutorial frames AutoML as meta-level learning and optimization around a learning system. In this lab, that learning system is a tabular ML pipeline, and FLAML searches over candidate learners and hyperparameters under a time budget.

## Three-Hour Plan

| Time | Activity |
| --- | --- |
| 0:00-0:20 | Reading warm-up from `hutter-vanschoren-part1-2.pdf` |
| 0:20-0:55 | Real classification task: Adult Census Income |
| 0:55-1:30 | FLAML for classification and metric choice |
| 1:30-1:40 | Break |
| 1:40-2:15 | Real regression task: California Housing |
| 2:15-2:45 | Budget comparison and model inspection |
| 2:45-3:00 | Deliverables and wrap-up discussion |

## Reading Warm-Up

Open `slides/resources/hutter-vanschoren-part1-2.pdf` and skim the first two parts of the tutorial.

Focus on four ideas:

- AutoML can be formulated as hyperparameter optimization with a top-level algorithm choice.
- Black-box HPO is expensive because each evaluation trains and validates a model.
- Random search is a surprisingly strong baseline when some dimensions matter more than others.
- Multi-fidelity methods save time by allocating more resources to promising configurations.

**Discussion question:** if AutoML returns a high-scoring model, what evidence do you still need before trusting it?

## AutoML as Hyperparameter Optimization

![](img/automl/hutter-vanschoren-11.png)

The algorithm choice can be treated as a hyperparameter. Once the algorithm is chosen, many other hyperparameters become conditional: a random forest has tree-related parameters, while logistic regression has regularization-related parameters.

## Why Budgets Matter

![](img/automl/hutter-vanschoren-13.png)

A single evaluation means training and validating a candidate model. On real data, that can be slow. FLAML is designed for economical AutoML: it tries to find good models under explicit resource constraints.

## 1. Setup

If you run this lab in Google Colab or a minimal Python environment, run the following cell first.

The setup cell installs only missing packages. The primary datasets are real public datasets; if network access is unavailable, the notebook falls back to real built-in scikit-learn datasets so the lab remains runnable.

In [ ]:
#| echo: false
# Install only packages missing from the current runtime.
import importlib.util
import subprocess
import sys

required_packages = ["flaml", "matplotlib", "numpy", "pandas", "sklearn"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    pip_names = []
    for pkg in missing_packages:
        if pkg == "sklearn":
            pip_names.append("scikit-learn")
        elif pkg == "flaml":
            pip_names.append("flaml[automl]")
        else:
            pip_names.append(pkg)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_names])

In [ ]:
#| echo: false
# Import data, modeling, preprocessing, evaluation, and plotting tools.
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_california_housing, fetch_openml, load_diabetes
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
#| echo: false
# Keep OneHotEncoder compatible with recent and older scikit-learn versions.
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def evaluate_classifier(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    else:
        y_score = y_pred
    return {
        "accuracy": accuracy_score(y_eval, y_pred),
        "roc_auc": roc_auc_score(y_eval, y_score),
        "f1": f1_score(y_eval, y_pred),
    }


def evaluate_regressor(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "rmse": mean_squared_error(y_eval, y_pred, squared=False),
        "mae": mean_absolute_error(y_eval, y_pred),
        "r2": r2_score(y_eval, y_pred),
    }


def display_scores(label, scores):
    print(label)
    for name, value in scores.items():
        print(f"  {name:8s}: {value:.3f}")

## 2. Classification Dataset: Adult Census Income

The first real dataset is Adult Census Income from the UCI repository, loaded through OpenML.

The task is to predict whether a person earns more than 50K USD per year from demographic and work-related attributes.

This is a better AutoML teaching case than a tiny built-in dataset because it has mixed numerical and categorical variables, missing values, class imbalance, and a socially sensitive target.

In [ ]:
def load_adult_income(n_rows=8000):
    try:
        adult = fetch_openml("adult", version=2, as_frame=True)
        df = adult.frame.copy()
        df.columns = [c.strip().lower().replace("-", "_") for c in df.columns]
        if "class" in df.columns:
            df = df.rename(columns={"class": "income"})
        df = df.replace("?", np.nan).dropna()
        df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
        df = df.sample(n=min(n_rows, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
        return df, "OpenML Adult Census Income"
    except Exception as exc:
        from sklearn.datasets import load_breast_cancer

        print(f"Could not download Adult from OpenML: {exc}")
        print("Falling back to the real built-in breast cancer dataset.")
        cancer = load_breast_cancer(as_frame=True)
        df = cancer.data.copy()
        df["income"] = cancer.target
        return df, "scikit-learn breast cancer fallback"


adult_df, adult_source = load_adult_income()
print(adult_source)
display(adult_df.head())
display(adult_df["income"].value_counts(normalize=True).rename("target_share").round(3))

In [ ]:
classification_target = "income"
X_cls = adult_df.drop(columns=[classification_target])
y_raw = adult_df[classification_target]

if y_raw.dtype == object:
    y_cls = (y_raw == ">50K").astype(int)
else:
    y_cls = y_raw.astype(int)

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_cls,
)

numeric_cls = X_cls_train.select_dtypes(include=np.number).columns.tolist()
categorical_cls = [c for c in X_cls_train.columns if c not in numeric_cls]

print(f"Rows: {len(X_cls)}")
print(f"Features: {X_cls.shape[1]}")
print(f"Numerical features: {len(numeric_cls)}")
print(f"Categorical features: {len(categorical_cls)}")
print(f"Positive class share: {y_cls.mean():.3f}")

## 3. Manual Classification Baseline

We build a baseline with explicit preprocessing:

- numeric features are scaled;
- categorical features are one-hot encoded;
- logistic regression predicts the income label.

This baseline is intentionally simple, but it is strong enough to make the AutoML comparison meaningful.

In [ ]:
classification_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cls),
        ("cat", make_one_hot_encoder(), categorical_cls),
    ]
)

classification_baseline = Pipeline(
    steps=[
        ("preprocess", classification_preprocess),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

classification_baseline.fit(X_cls_train, y_cls_train)
classification_baseline_scores = evaluate_classifier(classification_baseline, X_cls_test, y_cls_test)
display_scores("Manual classification baseline", classification_baseline_scores)
print("confusion matrix:")
print(confusion_matrix(y_cls_test, classification_baseline.predict(X_cls_test)))

## 4. FLAML for Classification

FLAML can work with pandas data directly, including mixed tabular inputs. We give it the same train/test split and reserve the test set for final evaluation.

The first AutoML run optimizes ROC AUC for 60 seconds.

In [ ]:
classification_automl = AutoML()

classification_automl.fit(
    X_train=X_cls_train,
    y_train=y_cls_train,
    task="classification",
    metric="roc_auc",
    time_budget=60,
    log_file_name="flaml_adult_roc_auc_60s.log",
    seed=RANDOM_STATE,
    verbose=1,
)

In [ ]:
print(f"Best estimator: {classification_automl.best_estimator}")
print(f"Best validation loss: {classification_automl.best_loss:.6f}")
print(f"Training time for best config: {classification_automl.best_config_train_time:.3f}s")
classification_automl.best_config

In [ ]:
classification_automl_scores = evaluate_classifier(classification_automl, X_cls_test, y_cls_test)
display_scores("FLAML classification model", classification_automl_scores)
print("confusion matrix:")
print(confusion_matrix(y_cls_test, classification_automl.predict(X_cls_test)))

pd.DataFrame(
    [classification_baseline_scores, classification_automl_scores],
    index=["manual_baseline", "flaml"],
).round(3)

## Random Search Is a Real Baseline

![](img/automl/hutter-vanschoren-14.png)

The tutorial emphasizes that random search can outperform grid search when only a few hyperparameters really matter. AutoML systems should therefore be compared against sensible baselines, not just against exhaustive grid search.

## 5. Classification Metrics

On Adult Income, accuracy can look good even when the model performs poorly on the minority class.

Run the same AutoML procedure with different optimization metrics and compare the final test results.

In [ ]:
def run_flaml_classification(metric="roc_auc", time_budget=45):
    model = AutoML()
    started_at = time.time()
    model.fit(
        X_train=X_cls_train,
        y_train=y_cls_train,
        task="classification",
        metric=metric,
        time_budget=time_budget,
        log_file_name=f"flaml_adult_{metric}_{time_budget}s.log",
        seed=RANDOM_STATE,
        verbose=0,
    )
    elapsed = time.time() - started_at
    scores = evaluate_classifier(model, X_cls_test, y_cls_test)
    return {
        "metric": metric,
        "budget": time_budget,
        "elapsed": elapsed,
        "best_estimator": model.best_estimator,
        "best_loss": model.best_loss,
        **scores,
    }


classification_metric_results = pd.DataFrame(
    [run_flaml_classification(metric=m, time_budget=45) for m in ["accuracy", "roc_auc", "f1"]]
)
classification_metric_results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
classification_metric_results.set_index("metric")[["accuracy", "roc_auc", "f1"]].plot(kind="bar", ax=ax)
ax.set_title("Adult Income: test scores for different FLAML objectives")
ax.set_xlabel("optimized metric")
ax.set_ylabel("test score")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.show()

### Classification Questions

1. Which metric selected the strongest model for ROC AUC?
2. Which metric selected the strongest model for F1?
3. Which errors are more costly in this task?
4. What additional fairness or governance checks would you run before deployment?

## 6. Regression Dataset: California Housing

The second real dataset is California Housing, originally derived from the 1990 US census.

The task is to predict median house value for California districts from demographic and geographic attributes.

This gives us a second AutoML setting: regression rather than classification.

In [ ]:
def load_housing_data():
    try:
        housing = fetch_california_housing(as_frame=True)
        return housing.frame.copy(), "California Housing"
    except Exception as exc:
        print(f"Could not download California Housing: {exc}")
        print("Falling back to the real built-in diabetes regression dataset.")
        diabetes = load_diabetes(as_frame=True)
        df = diabetes.frame.copy()
        df = df.rename(columns={"target": "MedHouseVal"})
        return df, "scikit-learn diabetes fallback"


housing_df, housing_source = load_housing_data()
print(housing_source)
display(housing_df.head())
display(housing_df.describe().T[["mean", "std", "min", "max"]].round(3))

In [ ]:
regression_target = "MedHouseVal" if "MedHouseVal" in housing_df.columns else "target"
X_reg = housing_df.drop(columns=[regression_target])
y_reg = housing_df[regression_target]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print(f"Rows: {len(X_reg)}")
print(f"Features: {X_reg.shape[1]}")
print(f"Target: {regression_target}")

## 7. Manual Regression Baselines

We compare FLAML against two common baselines:

- Ridge regression: simple, fast, and linear.
- Random forest: nonlinear and stronger, but less transparent and more expensive.

In [ ]:
ridge_baseline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=1.0)),
    ]
)
forest_baseline = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

ridge_baseline.fit(X_reg_train, y_reg_train)
forest_baseline.fit(X_reg_train, y_reg_train)

ridge_scores = evaluate_regressor(ridge_baseline, X_reg_test, y_reg_test)
forest_scores = evaluate_regressor(forest_baseline, X_reg_test, y_reg_test)

display_scores("Ridge regression baseline", ridge_scores)
display_scores("Random forest baseline", forest_scores)

pd.DataFrame([ridge_scores, forest_scores], index=["ridge", "random_forest"]).round(3)

## 8. FLAML for Regression

For regression, FLAML can optimize metrics such as RMSE, MAE, or R2.

Here we optimize RMSE and compare the selected model with the two manual baselines.

In [ ]:
regression_automl = AutoML()

regression_automl.fit(
    X_train=X_reg_train,
    y_train=y_reg_train,
    task="regression",
    metric="rmse",
    time_budget=60,
    log_file_name="flaml_housing_rmse_60s.log",
    seed=RANDOM_STATE,
    verbose=1,
)

In [ ]:
print(f"Best estimator: {regression_automl.best_estimator}")
print(f"Best validation loss: {regression_automl.best_loss:.6f}")
print(f"Training time for best config: {regression_automl.best_config_train_time:.3f}s")
regression_automl.best_config

In [ ]:
regression_automl_scores = evaluate_regressor(regression_automl, X_reg_test, y_reg_test)
display_scores("FLAML regression model", regression_automl_scores)

pd.DataFrame(
    [ridge_scores, forest_scores, regression_automl_scores],
    index=["ridge", "random_forest", "flaml"],
).round(3)

## Multi-Fidelity Search

![](img/automl/hutter-vanschoren-27.png)

A multi-fidelity strategy evaluates many configurations cheaply, then spends more resources on promising ones. FLAML's design is in the same spirit: good anytime behavior matters because the search may be stopped by a fixed budget.

## 9. Budget Experiment

Repeat FLAML on the regression task with short, medium, and longer budgets.

Watch whether validation loss and test-set performance move together.

In [ ]:
def run_flaml_regression(time_budget=45, metric="rmse"):
    model = AutoML()
    started_at = time.time()
    model.fit(
        X_train=X_reg_train,
        y_train=y_reg_train,
        task="regression",
        metric=metric,
        time_budget=time_budget,
        log_file_name=f"flaml_housing_{metric}_{time_budget}s.log",
        seed=RANDOM_STATE,
        verbose=0,
    )
    elapsed = time.time() - started_at
    scores = evaluate_regressor(model, X_reg_test, y_reg_test)
    return {
        "budget": time_budget,
        "metric": metric,
        "elapsed": elapsed,
        "best_estimator": model.best_estimator,
        "best_loss": model.best_loss,
        **scores,
    }


regression_budget_results = pd.DataFrame(
    [run_flaml_regression(time_budget=b, metric="rmse") for b in [15, 45, 120]]
)
regression_budget_results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(regression_budget_results["budget"], regression_budget_results["rmse"], marker="o", label="test RMSE")
ax.plot(regression_budget_results["budget"], regression_budget_results["best_loss"], marker="o", label="validation RMSE")
ax.set_xlabel("time budget (seconds)")
ax.set_ylabel("RMSE")
ax.set_title("California Housing: validation and test RMSE across budgets")
ax.legend()
plt.show()

## Combining Search Ideas

![](img/automl/hutter-vanschoren-34.png)

The Hutter and Vanschoren tutorial presents combinations of Bayesian optimization and multi-fidelity methods as a way to get both good anytime behavior and strong final performance.

In practice, this is why the budget is a modeling decision: a system that is excellent after two hours may not be the best choice for an interactive workflow that allows only two minutes.

## 10. Model Inspection and Reproducibility

For each FLAML run, record:

- dataset name and version;
- feature preprocessing and missing-value handling;
- train/test split and random seed;
- task type and optimized metric;
- time budget;
- selected learner;
- best validation loss;
- best configuration;
- final test-set metrics.

AutoML is most useful when its choices are inspectable and reproducible.

In [ ]:
summary = pd.DataFrame(
    [
        {
            "task": "classification",
            "dataset": adult_source,
            "optimized_metric": "roc_auc",
            "best_estimator": classification_automl.best_estimator,
            "best_loss": classification_automl.best_loss,
            **classification_automl_scores,
        },
        {
            "task": "regression",
            "dataset": housing_source,
            "optimized_metric": "rmse",
            "best_estimator": regression_automl.best_estimator,
            "best_loss": regression_automl.best_loss,
            **regression_automl_scores,
        },
    ]
)
summary.round(3)

## What AutoML Automates

![](img/automl/hutter-vanschoren-60.png)

AutoML can automate repetitive parts of modeling: trying candidate learners, allocating trials, tuning hyperparameters, and returning a strong baseline.

It does not automate the full data-science responsibility: problem formulation, target definition, leakage checks, fairness assessment, deployment monitoring, and deciding whether a model should be used at all.

## Deliverables

Submit a short lab report with:

1. A one-paragraph summary of AutoML based on `hutter-vanschoren-part1-2.pdf`.
2. The manual and FLAML results for Adult Census Income.
3. The manual and FLAML results for California Housing.
4. The metric-choice comparison for classification.
5. The budget comparison for regression.
6. A short answer to: **What did AutoML automate here, and what did it not automate?**

## Extension Tasks

Choose one if you finish early:

- Add `estimator_list=["lgbm", "xgboost", "rf", "extra_tree"]` after installing the optional learners.
- Save and reload an AutoML run with `automl.pickle()` and `AutoML.load_pickle()`.
- Repeat the classification task with a fairness-sensitive subgroup comparison.
- Create a custom metric that penalizes false negatives more heavily than false positives.
- Compare FLAML with a manual `RandomizedSearchCV` baseline.

## References

- Hutter, F., and Vanschoren, J. *Automatic Machine Learning (AutoML): A Tutorial*, NeurIPS 2018, Parts 1-2.
- Hutter, F., Kotthoff, L., and Vanschoren, J. (eds.). *Automated Machine Learning: Methods, Systems, Challenges*. Springer, 2019.
- FLAML documentation: Task-Oriented AutoML.
- FLAML documentation: AutoML Classification and Regression examples.
- OpenML Adult Census Income dataset.
- scikit-learn California Housing dataset.